In [1]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.8 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)
print(dataset)

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [3]:
import torch
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)

PyTorch: 2.11.0+cu130
CUDA available: True
bitsandbytes: 0.50.2


In [4]:
print("Columns:", dataset["train"].column_names)
print("\nFirst example:")
print(dataset["train"][0])

Columns: ['flags', 'instruction', 'category', 'intent', 'response']

First example:
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [5]:
def format_example(example):
  return{
      "messages":[
          {
              "role":"user",
              "content":example["instruction"]
          },
          {
              "role":"assistant",
              "content":example["response"]
          }
      ]
  }
formatted_dataset = dataset["train"].map(format_example)
print(formatted_dataset[0]["messages"])

Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

[{'content': 'question about cancelling order {{Order Number}}', 'role': 'user'}, {'content': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.", 'role': 'assistant'}]


In [6]:
split_1= formatted_dataset.train_test_split(test_size=0.2,seed=42)
split_2= split_1["test"].train_test_split(test_size=0.5,seed=42)

train_dataset = split_1["train"]
validation_dataset = split_2["train"]
test_dataset = split_2["test"]

print("Training:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Testing:", len(test_dataset))

Training: 21497
Validation: 2687
Testing: 2688


In [7]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)

print("Hugging Face login successful!")

Hugging Face login successful!


In [8]:
from huggingface_hub import model_info

model_id = "meta-llama/Llama-3.2-1B-Instruct"
info = model_info(model_id, token=hf_token)

print("Model:", info.id)
print("Access verified!")

Model: meta-llama/Llama-3.2-1B-Instruct
Access verified!


In [9]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
quant_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(model_id,token=hf_token,quantization_config=quant_config,device_map="auto")

print("Tokenizer and model loaded successfully!")
print("Chat template available:", tokenizer.chat_template is not None)
print("GPU:", torch.cuda.get_device_name(0))


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Tokenizer and model loaded successfully!
Chat template available: True
GPU: Tesla T4


In [10]:
messages = [
    {"role": "user", "content": "How can I cancel my order?"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Base model response:\n")
print(response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Base model response:

If you're looking to cancel your order, I'll need more information about the specific order you're referring to. Could you please provide me with the following details:

1. Order number or order ID
2. The order you're trying to cancel (if you have it)
3. The reason for cancellation (if you want to provide a reason)

Once I have this information, I can guide you through the process of canceling your order.


In [11]:
def format_for_training(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_formatted = train_dataset.map(format_for_training)
validation_formatted = validation_dataset.map(format_for_training)

print(train_formatted[0]["text"])

Map:   0%|          | 0/21497 [00:00<?, ? examples/s]

Map:   0%|          | 0/2687 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 21 Oct 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

I am trying to delete some items from order {{Order Number}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Thank you for bringing this to our attention. I understand that you would like to delete some items from order number {{Order Number}}. We apologize for any inconvenience this may have caused. To facilitate the deletion process, could you please provide the specific details of the items you wish to remove? Once we receive the information, we will work swiftly to ensure the necessary modifications are made. Our ultimate goal is to provide you with a seamless and satisfactory shopping experience.<|eot_id|>


In [12]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
  )
print("QLoRA configuration ready!")
print("Trainable adapter settings:", lora_config)

QLoRA configuration ready!
Trainable adapter settings: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'k_proj', 'v_proj', 'down_proj', 'q_proj', 'gate_proj', 'o_proj', 'up_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None,

In [13]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./SupportSage_results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,
    dataset_text_field="text",
    remove_unused_columns=False
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_formatted,
    eval_dataset=validation_formatted,
    peft_config=lora_config,
    processing_class=tokenizer
)

print("Trainer created successfully!")

Tokenizing train dataset:   0%|          | 0/21497 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/21497 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/21497 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/21497 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2687 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/2687 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2687 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/2687 [00:00<?, ? examples/s]

Trainer created successfully!


In [14]:
trainer.model.print_trainable_parameters()
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"Allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

trainable params: 5,636,096 || all params: 1,241,450,496 || trainable%: 0.4540

GPU: Tesla T4
Allocated memory: 1.51 GB
Reserved memory: 2.04 GB


In [ ]:
train_result = trainer.train()

print("\nFine-tuning completed!")
print(train_result.metrics)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
200,0.711370,0.729093,0.729927,0.785764,272817.000000
400,0.680859,0.660331,0.699954,0.801493,547187.000000
600,0.636059,0.632091,0.645096,0.807167,816199.000000
800,0.586923,0.612463,0.600353,0.811162,1090502.000000
1000,0.588923,0.599895,0.601712,0.813423,1364336.000000
1200,0.593631,0.586531,0.590670,0.816600,1641324.000000
1400,0.563608,0.578583,0.569242,0.818947,1915464.000000
1600,0.569951,0.571435,0.570183,0.820520,2189801.000000
1800,0.569405,0.562114,0.561029,0.822682,2465291.000000
2000,0.575276,0.557320,0.560690,0.823992,2736250.000000



Fine-tuning completed!
{'train_runtime': 11245.684, 'train_samples_per_second': 1.912, 'train_steps_per_second': 0.239, 'total_flos': 2.5933737038155776e+16, 'train_loss': 0.6148933319463616, 'epoch': 1.0}


In [15]:
from google.colab import drive
import os

# Connect Google Drive
drive.mount("/content/drive")

# Choose a permanent location in Google Drive
save_path = "/content/drive/MyDrive/SupportSage_AI/final_adapter"
os.makedirs(save_path, exist_ok=True)

# Save the trained LoRA adapter and tokenizer
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Fine-tuned adapter saved successfully!")
print("Location:", save_path)

Mounted at /content/drive
Fine-tuned adapter saved successfully!
Location: /content/drive/MyDrive/SupportSage_AI/final_adapter


In [16]:
from google.colab import drive
drive.mount("/content/drive")
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_id = "meta-llama/Llama-3.2-1B-Instruct"
adapter_path = "/content/drive/MyDrive/SupportSage_AI/final_adapter"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
quant_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16)
base_model = AutoModelForCausalLM.from_pretrained(model_id,token=hf_token,quantization_config=quant_config,device_map="auto")
model = PeftModel.from_pretrained(base_model, adapter_path)

model.eval()

print("Fine-tuned model loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Fine-tuned model loaded successfully!


In [18]:

def chat_with_supportsage(question):
    messages = [
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Move input tensors to the model's device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[-1]
    response = outputs[0][input_length:]

    return tokenizer.decode(response, skip_special_tokens=True)


print(chat_with_supportsage(
    "I want to cancel my order. How can I do that?"
))


To cancel your order, you can try the following steps:

1. **Contact the seller directly**: Reach out to the company that sent you the order and ask to cancel it. Be prepared to provide your order number and any other relevant information.
2. **Check the seller's website**: Look for a "Cancel Order" or "Change Order" option on the seller's website. If you can't find it, you can try searching for the option or contacting their customer service department.
3. **Use the seller's cancellation process**: Some sellers have a dedicated cancellation process on their website. You can check the website to see if this is the case.
4. **Reach out to the shipping carrier**: If you ordered shipping, you can contact
